# LogLens: QLoRA fine-tuning of Qwen2.5-1.5B-Instruct

Runs on a free **Colab T4** or **Kaggle T4/P100**. This notebook calls `training/train_qlora.py`, so the notebook and the script stay identical.

Before you start:
1. Colab: *Runtime > Change runtime type > T4 GPU*. Kaggle: *Settings > Accelerator > GPU T4 x2* and turn Internet on.
2. Have `train_chat.jsonl` and `val_chat.jsonl` from `data/processed/` ready to upload.
3. (Optional, for pushing to the Hub) add a Hugging Face token with write access as a Colab secret named `HF_TOKEN`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/Saipavanavsp/loglens.git"
!git clone --depth 1 {REPO_URL} loglens
%cd loglens/training
!pip install -q -r requirements.txt

## Data
Upload the two chat files. On Kaggle, attach them as a dataset instead and set `DATA_DIR` to its path.

In [ ]:
import os
import shutil
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
try:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, DATA_DIR / name)
except ImportError:
    DATA_DIR = Path("/kaggle/input/loglens-data")  # edit to your Kaggle dataset path
print(sorted(p.name for p in DATA_DIR.iterdir()))
!wc -l {DATA_DIR}/*.jsonl

## Config
Defaults are explained in `train_qlora.py` and `docs/FINETUNING.md`. Change fields here if needed.

In [ ]:
from dataclasses import asdict

from train_qlora import TrainConfig, train

cfg = TrainConfig()
# cfg.num_epochs = 2  # example override
asdict(cfg)

In [ ]:
OUT = Path("outputs/run1")
OUT.mkdir(parents=True, exist_ok=True)
summary = train(cfg, DATA_DIR, OUT)

## Loss curve

In [ ]:
import matplotlib.pyplot as plt

hist = summary["log_history"]
train_pts = [h for h in hist if "loss" in h]
val_pts = [h for h in hist if "eval_loss" in h]
plt.plot([h["step"] for h in train_pts], [h["loss"] for h in train_pts], label="train")
plt.plot([h["step"] for h in val_pts], [h["eval_loss"] for h in val_pts], "o-", label="val")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.title("QLoRA loss")
plt.savefig(OUT / "loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## Merge and export to GGUF (for Ollama)
Builds llama.cpp (about 5 minutes), merges the adapter, converts to GGUF and quantises to Q4_K_M.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
%cd /content/loglens/training

In [ ]:
HF_REPO = ""  # e.g. "your-hf-user/loglens-qwen2.5-1.5b" (private); leave empty to skip upload
if HF_REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
push = f"--hf-repo {HF_REPO}" if HF_REPO else ""
!python merge_and_export.py --adapter {OUT}/adapter --out {OUT} --llama-cpp /content/llama.cpp {push}

## Download results
Download `training_summary.json` and `loss_curve.png`, plus the GGUF if you didn't push it to the Hub. Put them in `reports/training/` locally and give the numbers to the next step.

In [ ]:
!cd {OUT} && zip -q -r /content/loglens_run1.zip training_summary.json loss_curve.png adapter
try:
    from google.colab import files
    files.download("/content/loglens_run1.zip")
    files.download(str(OUT / "loglens-q4_k_m.gguf"))
except ImportError:
    print("On Kaggle: download from the Output tab")